# Pandas 从入门到数据分析

这份专题不是 API 大全，而是一条完整的数据分析主线：

```text
读取数据 -> 认识数据 -> 筛选 -> 清洗 -> 计算
        -> 分组汇总 -> 表连接 -> 透视分析 -> 导出
```

学习分三层：

- **必学**：DataFrame、查看、选择、筛选、缺失值、排序。
- **实战**：分组统计、表连接、日期、透视表。
- **进阶**：方法链、性能意识、Copy-on-Write 和常见面试题。

建议分两次学习，每次 40～60 分钟。

## 0. Pandas 是什么？

Pandas 是 Python 中常用的数据处理和分析库，特别适合处理**二维表格数据**。

常见应用：

- 读取 CSV、Excel；
- 清洗缺失值和重复数据；
- 按条件筛选记录；
- 分组统计；
- 多张表关联；
- 生成数据分析结果。

Pandas 不会取代数据库。数据量很大或需要多人并发查询时，通常仍由数据库完成主要计算，再用 Pandas 做进一步分析。

## 1. 导入 Pandas 与版本说明

约定俗成的导入方式：

```python
import pandas as pd
```

本专题在本机 `pandas 2.3.3` 上验证。

Pandas 3.0 默认启用 Copy-on-Write；在 2.3 中可以手动开启。它让切片和复制后的修改行为更可预测。本专题仍然推荐显式使用 `.loc` 修改原表，不依赖容易混淆的链式赋值。

In [ ]:
import pandas as pd

print("pandas 版本：", pd.__version__)

# 在 pandas 2.x 中主动启用；pandas 3.x 已默认启用。
if hasattr(pd.options.mode, "copy_on_write"):
    pd.options.mode.copy_on_write = True

# 第一部分：Series 与 DataFrame（必学）

## 2. 两种核心对象

### Series

一列带索引的数据，可以理解成“有标签的一维数组”。

### DataFrame

多列组成的二维表格。每一列都是一个 Series。

```text
DataFrame
├── order_id   -> Series
├── city       -> Series
├── quantity   -> Series
└── price      -> Series
```

专业名称：**标签对齐（Label Alignment）**。Pandas 做运算时会关注行列标签，不只看位置。

In [ ]:
scores = pd.Series([90, 85, 92], index=["小明", "小红", "小刚"], name="score")

print(scores)
print("小红的分数：", scores.loc["小红"])

assert scores.loc["小红"] == 85

## 3. 创建贯穿全篇的订单数据

我们使用一份小型订单表：

- 每一行代表一笔订单；
- 每一列代表订单的一个属性；
- `satisfaction` 中故意放入缺失值，方便练习数据清洗。

In [ ]:
raw_orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "customer_id": ["C01", "C02", "C01", "C03", "C04", "C02", "C05", "C03"],
    "city": ["上海", "北京", "上海", "广州", "北京", "北京", "深圳", "广州"],
    "category": ["电脑", "图书", "配件", "电脑", "图书", "配件", "电脑", "图书"],
    "quantity": [1, 3, 2, 1, 5, 4, 1, 2],
    "unit_price": [6999.0, 68.0, 199.0, 5899.0, 45.0, 129.0, 7999.0, 88.0],
    "order_date": [
        "2026-07-01", "2026-07-01", "2026-07-02", "2026-07-03",
        "2026-07-03", "2026-07-04", "2026-07-05", "2026-07-05",
    ],
    "satisfaction": [5, 4, None, 5, 3, None, 4, 5],
})

orders = raw_orders.copy()
orders

## 4. 拿到数据后先看什么？

常用检查：

| 写法 | 作用 |
|---|---|
| `df.head()` | 查看前 5 行 |
| `df.tail()` | 查看后 5 行 |
| `df.shape` | `(行数, 列数)` |
| `df.columns` | 列名 |
| `df.dtypes` | 每列数据类型 |
| `df.info()` | 行数、类型、非空数量、内存概览 |
| `df.describe()` | 数值列描述统计 |

分析前不要急着写复杂逻辑。先确认列名、类型、缺失值和数据规模。

In [ ]:
print("形状：", orders.shape)
print("列名：", orders.columns.tolist())
print("\n数据类型：")
print(orders.dtypes)
print("\n数值列描述统计：")
print(orders.describe())

assert orders.shape == (8, 8)

## 5. 索引 Index

DataFrame 左侧的 `0, 1, 2...` 是索引，不是普通数据列。

```python
df.index
df.set_index("order_id")
df.reset_index()
```

索引适合表示行标签，但不建议一入门就把所有字段都设成索引。普通分析中保留默认整数索引通常更直观。

注意：索引不要求天然唯一。业务主键是否唯一，需要自己检查：

```python
df["order_id"].is_unique
```

In [ ]:
indexed_orders = orders.set_index("order_id")

print(indexed_orders.head(2))
print("order_id 是否唯一：", orders["order_id"].is_unique)

assert indexed_orders.loc[1001, "city"] == "上海"
assert orders["order_id"].is_unique

# 第二部分：选择、筛选与修改（必学）

## 6. 选择一列和多列

选择一列：

```python
orders["city"]
```

返回 `Series`。

选择多列：

```python
orders[["city", "category"]]
```

返回 `DataFrame`。

注意双层方括号：外层表示选择操作，内层列表保存多个列名。

In [ ]:
city_series = orders["city"]
small_table = orders[["order_id", "city", "category"]]

print(type(city_series).__name__)
print(type(small_table).__name__)
print(small_table.head())

assert isinstance(city_series, pd.Series)
assert isinstance(small_table, pd.DataFrame)

## 7. `.loc` 与 `.iloc`

### `.loc`：按标签选择

```python
df.loc[行标签, 列标签]
```

### `.iloc`：按位置选择

```python
df.iloc[行位置, 列位置]
```

例子：

```python
orders.loc[0:2, ["city", "quantity"]]
orders.iloc[0:3, 2:5]
```

容易混淆的细节：

- `.loc[0:2]` 通常包含标签 2；
- `.iloc[0:3]` 和 Python 切片一样，不包含位置 3。

In [ ]:
by_label = orders.loc[0:2, ["city", "quantity"]]
by_position = orders.iloc[0:3, 2:5]

print(by_label)
print(by_position)

assert len(by_label) == 3
assert len(by_position) == 3

## 8. 条件筛选

筛选北京订单：

```python
orders[orders["city"] == "北京"]
```

多个条件：

```python
orders[(orders["city"] == "北京") & (orders["quantity"] >= 3)]
```

规则：

- 与：`&`
- 或：`|`
- 非：`~`
- 每个条件分别加括号；
- 不要使用普通 Python 的 `and`、`or` 连接 Series 条件。

匹配多个值：

```python
orders[orders["city"].isin(["北京", "上海"])]
```

In [ ]:
beijing_large = orders[
    (orders["city"] == "北京") & (orders["quantity"] >= 3)
]

selected_cities = orders[orders["city"].isin(["北京", "上海"])]

print(beijing_large[["order_id", "city", "quantity"]])
assert beijing_large["order_id"].tolist() == [1002, 1005, 1006]
assert set(selected_cities["city"]) == {"北京", "上海"}

## 9. 新增和修改列

新增订单金额：

```python
orders["amount"] = orders["quantity"] * orders["unit_price"]
```

条件修改推荐 `.loc`：

```python
orders.loc[orders["amount"] >= 5000, "level"] = "高金额"
```

不要写链式赋值：

```python
orders[orders["amount"] >= 5000]["level"] = "高金额"  # 不推荐
```

链式赋值可能修改的是临时对象，而且在 Copy-on-Write 模式下无法达到修改原表的目的。

In [ ]:
orders = raw_orders.copy()
orders["amount"] = orders["quantity"] * orders["unit_price"]
orders["level"] = "普通"
orders.loc[orders["amount"] >= 5000, "level"] = "高金额"

print(orders[["order_id", "amount", "level"]])
assert orders.loc[orders["order_id"] == 1001, "amount"].iloc[0] == 6999.0

## 10. 删除、改名与排序

```python
df.drop(columns=["level"])
df.rename(columns={"unit_price": "price"})
df.sort_values("amount", ascending=False)
df.sort_values(["city", "amount"], ascending=[True, False])
```

这些方法默认返回新 DataFrame。推荐显式接收结果：

```python
sorted_orders = orders.sort_values("amount", ascending=False)
```

比到处使用 `inplace=True` 更容易跟踪数据变化。

In [ ]:
sorted_orders = orders.sort_values("amount", ascending=False)
renamed_orders = orders.rename(columns={"unit_price": "price"})
without_level = orders.drop(columns=["level"])

print(sorted_orders[["order_id", "amount"]].head(3))
assert sorted_orders.iloc[0]["order_id"] == 1007
assert "price" in renamed_orders.columns
assert "level" not in without_level.columns

# 第三部分：数据清洗（必学）

## 11. 缺失值

Pandas 中可能遇到 `NaN`、`None`、`pd.NA` 等缺失表示。

检查：

```python
df.isna()
df.isna().sum()
df["column"].notna()
```

处理：

```python
df.dropna()
df["column"].fillna(默认值)
```

不能可靠地使用 `value == None` 或 `value == float("nan")` 判断缺失，统一使用 `pd.isna()`、`.isna()`。

In [ ]:
print("每列缺失数量：")
print(orders.isna().sum())

clean_orders = orders.copy()
median_score = clean_orders["satisfaction"].median()
clean_orders["satisfaction"] = clean_orders["satisfaction"].fillna(median_score)

print("填充后的满意度：", clean_orders["satisfaction"].tolist())
assert clean_orders["satisfaction"].isna().sum() == 0

## 12. 重复值

检查完全重复的行：

```python
df.duplicated()
```

按照业务主键检查：

```python
df.duplicated(subset=["order_id"])
```

删除：

```python
df.drop_duplicates(subset=["order_id"], keep="first")
```

关键不是“看到重复就删”，而是先明确业务规则：同一个客户多次下单不是重复，同一个订单号出现两次才可能是重复数据。

In [ ]:
duplicated_orders = pd.concat([orders, orders.iloc[[0]]], ignore_index=True)

print("重复订单号数量：", duplicated_orders.duplicated(subset=["order_id"]).sum())
deduplicated = duplicated_orders.drop_duplicates(subset=["order_id"], keep="first")

assert len(duplicated_orders) == 9
assert len(deduplicated) == 8

## 13. 数据类型转换

常用转换：

```python
df["quantity"] = df["quantity"].astype("int64")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
```

`errors="coerce"` 会把无法转换的值变成缺失值，方便后续统一检查；但不能转换完就不管，应该再次检查 `.isna().sum()`。

In [ ]:
typed_orders = orders.copy()
typed_orders["order_date"] = pd.to_datetime(typed_orders["order_date"], errors="coerce")
typed_orders["quantity"] = typed_orders["quantity"].astype("int64")

print(typed_orders.dtypes)
assert str(typed_orders["order_date"].dtype).startswith("datetime64")

## 14. 字符串处理 `.str`

Pandas 为字符串列提供向量化字符串方法：

```python
df["name"].str.strip()
df["name"].str.lower()
df["name"].str.contains("关键字", na=False)
df["code"].str.split("-")
df["phone"].str[:3]
```

这些操作是对整列执行，不需要手写逐行循环。

In [ ]:
products = pd.Series(["  Python Book ", "Pandas Guide", None], dtype="string")

clean_products = products.str.strip().str.lower()
contains_python = clean_products.str.contains("python", na=False)

print(clean_products)
print(contains_python)
assert contains_python.tolist() == [True, False, False]

## 15. 日期处理 `.dt`

先转换为日期类型：

```python
df["order_date"] = pd.to_datetime(df["order_date"])
```

再提取：

```python
df["order_date"].dt.year
df["order_date"].dt.month
df["order_date"].dt.day
df["order_date"].dt.dayofweek
```

日期相减会得到时间差，可以继续使用 `.dt.days`。

In [ ]:
date_orders = orders.copy()
date_orders["order_date"] = pd.to_datetime(date_orders["order_date"])
date_orders["day"] = date_orders["order_date"].dt.day
date_orders["weekday"] = date_orders["order_date"].dt.day_name()

print(date_orders[["order_date", "day", "weekday"]].head())
assert date_orders["day"].tolist()[:3] == [1, 1, 2]

# 第四部分：统计分析（实战）

## 16. `value_counts`、`unique`、`nunique`

```python
df["city"].value_counts()           # 每个值出现次数
df["city"].value_counts(normalize=True)  # 占比
df["city"].unique()                 # 不重复值
df["city"].nunique()                # 不重复值数量
```

做分类字段探索时，这几项非常常用。

In [ ]:
city_counts = orders["city"].value_counts()
city_ratio = orders["city"].value_counts(normalize=True)

print(city_counts)
print(city_ratio)

assert city_counts["北京"] == 3
assert orders["city"].nunique() == 4

## 17. `groupby`：分组统计

核心思想叫 **拆分—应用—合并（Split-Apply-Combine）**：

1. 按城市拆成多组；
2. 每组计算订单数、销售额；
3. 合并成结果表。

```python
orders.groupby("city")["amount"].sum()
```

推荐使用命名聚合，让结果列名清楚：

```python
orders.groupby("city", as_index=False).agg(
    order_count=("order_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
)
```

In [ ]:
city_summary = (
    orders.groupby("city", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_amount=("amount", "sum"),
        avg_amount=("amount", "mean"),
    )
    .sort_values("total_amount", ascending=False)
)

print(city_summary)
assert city_summary["order_count"].sum() == len(orders)

## 18. `agg` 与 `transform` 的区别

### `agg`

把每组压缩成较少的汇总行。

### `transform`

计算分组结果后，再映射回原来的每一行，输出长度与原表相同。

例如计算每笔订单占所在城市销售额的比例：

```python
city_total = orders.groupby("city")["amount"].transform("sum")
orders["city_share"] = orders["amount"] / city_total
```

In [ ]:
share_orders = orders.copy()
share_orders["city_total"] = share_orders.groupby("city")["amount"].transform("sum")
share_orders["city_share"] = share_orders["amount"] / share_orders["city_total"]

print(share_orders[["order_id", "city", "amount", "city_share"]])

share_sum = share_orders.groupby("city")["city_share"].sum()
assert share_sum.round(10).eq(1).all()

## 19. 透视表 `pivot_table`

透视表适合从两个维度观察指标：

```python
pd.pivot_table(
    orders,
    index="city",
    columns="category",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)
```

- `index`：行维度；
- `columns`：列维度；
- `values`：要统计的指标；
- `aggfunc`：汇总方式。

In [ ]:
sales_pivot = pd.pivot_table(
    orders,
    index="city",
    columns="category",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)

print(sales_pivot)
assert sales_pivot.loc["上海", "电脑"] == 6999.0

# 第五部分：多表处理（实战）

## 20. `merge`：像 SQL JOIN 一样连接表

客户表：

```text
customer_id | customer_name | member_level
```

订单表通过 `customer_id` 与客户表关联。

```python
pd.merge(orders, customers, on="customer_id", how="left")
```

常见连接方式：

- `left`：保留左表全部记录；
- `inner`：只保留两边都匹配的记录；
- `outer`：保留两边所有记录。

业务分析中最常见的是 `left`，因为通常希望保留主表全部记录。

In [ ]:
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04", "C05"],
    "customer_name": ["小明", "小红", "小刚", "小李", "小周"],
    "member_level": ["金卡", "银卡", "金卡", "普通", "银卡"],
})

order_details = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print(order_details[["order_id", "customer_name", "member_level"]].head())
assert len(order_details) == len(orders)
assert order_details["customer_name"].isna().sum() == 0

## 21. 连接后为什么行数变多？

如果右表的连接键重复，一条左表记录可能匹配多条右表记录，结果行数就会增加。

建议连接前检查：

```python
customers["customer_id"].is_unique
```

并使用 `validate` 声明预期关系：

```python
validate="one_to_one"
validate="many_to_one"
validate="one_to_many"
```

关系不符合预期时，Pandas 会直接报错，比默默生成错误结果更安全。

## 22. `concat`：纵向或横向拼接

追加新批次数据：

```python
all_orders = pd.concat([july_orders, august_orders], ignore_index=True)
```

横向拼接：

```python
pd.concat([left_table, right_table], axis=1)
```

`concat` 不会像 `merge` 那样根据业务键自动匹配。纵向拼接时通常按列名对齐，横向拼接时按索引对齐。

In [ ]:
new_order = orders.iloc[[0]].copy()
new_order.loc[:, "order_id"] = 1009

all_orders = pd.concat([orders, new_order], ignore_index=True)

print(all_orders.tail(2))
assert len(all_orders) == 9
assert all_orders.iloc[-1]["order_id"] == 1009

# 第六部分：文件读写

## 23. CSV

读取：

```python
df = pd.read_csv(
    "orders.csv",
    encoding="utf-8",
    usecols=["order_id", "city", "amount"],
    dtype={"order_id": "string"},
    parse_dates=["order_date"],
)
```

写出：

```python
df.to_csv("result.csv", index=False, encoding="utf-8-sig")
```

`index=False` 可以避免把 DataFrame 索引额外写成一列。

In [ ]:
from io import StringIO

csv_text = """order_id,city,amount
1001,上海,6999
1002,北京,204
"""

csv_orders = pd.read_csv(StringIO(csv_text), dtype={"order_id": "string"})
print(csv_orders)

exported = csv_orders.to_csv(index=False)
assert "order_id,city,amount" in exported
assert str(csv_orders["order_id"].dtype).startswith("string")

## 24. Excel

读取：

```python
df = pd.read_excel("data.xlsx", sheet_name="订单表")
```

读取多个工作表：

```python
sheets = pd.read_excel("data.xlsx", sheet_name=None)
```

写出多个工作表：

```python
with pd.ExcelWriter("result.xlsx") as writer:
    detail.to_excel(writer, sheet_name="明细", index=False)
    summary.to_excel(writer, sheet_name="汇总", index=False)
```

Excel 读写通常需要 `openpyxl` 等引擎。正式处理前要确认表头所在行、合并单元格和日期格式。

# 第七部分：写得更稳、更快

## 25. 优先使用向量化操作

不推荐逐行循环：

```python
for i, row in orders.iterrows():
    orders.loc[i, "amount"] = row["quantity"] * row["unit_price"]
```

推荐整列运算：

```python
orders["amount"] = orders["quantity"] * orders["unit_price"]
```

Pandas 擅长批量列运算。`iterrows()` 和逐行 `apply(axis=1)` 通常更慢，只有逻辑确实无法方便向量化时再考虑。

In [ ]:
vectorized_orders = raw_orders.copy()
vectorized_orders["amount"] = (
    vectorized_orders["quantity"] * vectorized_orders["unit_price"]
)

assert vectorized_orders["amount"].tolist() == orders["amount"].tolist()
vectorized_orders[["order_id", "amount"]]

## 26. 方法链 Method Chaining

把多个步骤按顺序连接：

```python
result = (
    orders
    .assign(amount=lambda x: x["quantity"] * x["unit_price"])
    .query("amount >= 500")
    .groupby("city", as_index=False)
    .agg(total_amount=("amount", "sum"))
    .sort_values("total_amount", ascending=False)
)
```

优点是数据流清楚；缺点是链条过长时不方便调试。初学时可以先分步写，熟练后再组合。

In [ ]:
analysis_result = (
    raw_orders
    .assign(amount=lambda frame: frame["quantity"] * frame["unit_price"])
    .query("amount >= 500")
    .groupby("city", as_index=False)
    .agg(total_amount=("amount", "sum"))
    .sort_values("total_amount", ascending=False)
)

print(analysis_result)
assert analysis_result["total_amount"].sum() == orders.loc[orders["amount"] >= 500, "amount"].sum()

## 27. 常见错误清单

1. 选多列时忘记双层方括号。
2. 用 `and/or` 连接 Series 条件，应使用 `&/|` 并加括号。
3. 混淆 `.loc` 标签与 `.iloc` 位置。
4. 使用链式赋值，结果没有修改原表。
5. 读取数据后不检查 `dtypes`，数字列其实是字符串。
6. 用 `== None` 或 `== NaN` 判断缺失值。
7. `merge` 后不检查行数和键关系。
8. `groupby` 后忘记索引变化，后续列找不到。
9. 为简单列计算使用 `iterrows()`。
10. 导出 CSV 时忘记 `index=False`。
11. 看到重复行就直接删除，没有先定义业务主键。
12. 修改原表太多次，却没有保留原始数据副本。

# 第八部分：综合案例

## 28. 业务问题

使用订单表回答：

1. 每个城市有多少订单？
2. 每个城市的销售额是多少？
3. 每个城市的客单价是多少？
4. 哪个城市销售额最高？
5. 只保留订单数至少为 2 的城市。

专业指标：

```text
客单价 = 销售额 / 订单数
```

In [ ]:
city_report = (
    orders.groupby("city", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_amount=("amount", "sum"),
    )
    .assign(avg_order_value=lambda frame: frame["total_amount"] / frame["order_count"])
    .query("order_count >= 2")
    .sort_values("total_amount", ascending=False)
    .reset_index(drop=True)
)

print(city_report)
assert set(city_report["city"]) == {"北京", "上海", "广州"}
assert city_report.iloc[0]["city"] == "上海"

# 第九部分：专项练习

请使用 `orders` 完成：

1. 筛选数量大于等于 2 的订单。
2. 新增 `amount` 列。
3. 找出销售额最高的 3 笔订单。
4. 统计每个商品类别的订单数和销售额。
5. 用满意度中位数填充缺失值。
6. 计算每个客户的累计消费金额。
7. 将客户表连接到订单表，并检查是否存在未匹配客户。

In [ ]:
# Pandas 专项练习区
# 请先自己完成，再看参考答案。

practice_orders = raw_orders.copy()

quantity_at_least_two = None
top_three_orders = None
category_summary = None
filled_orders = None
customer_summary = None
merged_orders = None

## 专项练习参考答案

参考答案不是唯一写法。重点检查每一步的输入、输出形状和业务含义。

In [ ]:
answer_orders = raw_orders.copy()
answer_orders["amount"] = answer_orders["quantity"] * answer_orders["unit_price"]

quantity_at_least_two_answer = answer_orders[answer_orders["quantity"] >= 2]

top_three_orders_answer = answer_orders.nlargest(3, "amount")

category_summary_answer = (
    answer_orders.groupby("category", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_amount=("amount", "sum"),
    )
)

filled_orders_answer = answer_orders.copy()
filled_orders_answer["satisfaction"] = filled_orders_answer["satisfaction"].fillna(
    filled_orders_answer["satisfaction"].median()
)

customer_summary_answer = (
    answer_orders.groupby("customer_id", as_index=False)
    .agg(total_amount=("amount", "sum"))
    .sort_values("total_amount", ascending=False)
)

merged_orders_answer = answer_orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

assert len(quantity_at_least_two_answer) == 5
assert top_three_orders_answer["order_id"].tolist() == [1007, 1001, 1004]
assert category_summary_answer["order_count"].sum() == len(answer_orders)
assert filled_orders_answer["satisfaction"].isna().sum() == 0
assert customer_summary_answer["total_amount"].sum() == answer_orders["amount"].sum()
assert merged_orders_answer["customer_name"].isna().sum() == 0
print("Pandas 专项练习参考答案测试通过")

# 第十部分：秋招面试表达

## 29. 常见问题

### Series 和 DataFrame 的区别？

Series 是带索引的一维数据结构；DataFrame 是带行列标签的二维表格，每列通常是一个 Series。

### `.loc` 和 `.iloc` 的区别？

`.loc` 主要按标签选择，`.iloc` 按整数位置选择。

### `merge` 和 `concat` 的区别？

`merge` 根据键进行类似数据库 JOIN 的关联；`concat` 沿指定轴拼接对象，主要依赖列名或索引对齐。

### `agg` 和 `transform` 的区别？

`agg` 通常把每组压缩成汇总结果；`transform` 返回与原对象相同长度的结果，适合把组统计量映射回每一行。

### 如何优化 Pandas 代码？

优先使用向量化操作，减少 Python 层逐行循环；读取时限制列和数据类型；尽早筛选不需要的数据；连接前检查键唯一性；大数据量时考虑数据库、Polars、Dask 或 Spark 等工具。

## 30. 面试项目表达模板

> 我使用 Pandas 读取并检查原始数据，首先核对字段类型、缺失值、重复记录和业务主键。清洗后通过向量化方式生成衍生指标，再使用 groupby 和命名聚合完成分组统计。多表关联时采用 merge，并通过 validate 和关联前后的行数检查防止重复键造成数据膨胀。最终使用透视表或汇总表输出结果，并保留原始数据和中间校验记录。

真正面试时，要把模板中的步骤替换成自己实际做过的项目内容。

# 七天学习建议

| 天数 | 内容 | 最低目标 |
|---|---|---|
| Day 1 | Series、DataFrame、数据查看 | 能解释行、列、索引和类型 |
| Day 2 | 选择、筛选、新增列、排序 | 能独立写布尔筛选 |
| Day 3 | 缺失、重复、类型、字符串和日期 | 能完成一次基础清洗 |
| Day 4 | value_counts、groupby、agg、transform | 能做城市汇总 |
| Day 5 | merge、concat、pivot_table | 能连接两张表并检查结果 |
| Day 6 | CSV、Excel、向量化和排错 | 能完成读取到导出的流程 |
| Day 7 | 综合案例与专项练习 | 不看答案完成 7 项任务 |

每天 40～60 分钟。Pandas 的关键不是背方法，而是知道一张表经过每一步后“行数、列数、类型和业务含义”发生了什么。

# 最终自检

- [ ] 能解释 Series、DataFrame 和 Index。
- [ ] 能使用 `.loc`、`.iloc` 和布尔条件选择数据。
- [ ] 知道多条件筛选为什么使用 `&`、`|` 和括号。
- [ ] 会检查和处理缺失值、重复值、错误类型。
- [ ] 会使用字符串和日期访问器 `.str`、`.dt`。
- [ ] 会使用 `groupby`、命名聚合和 `transform`。
- [ ] 会使用 `merge`、`concat` 和透视表。
- [ ] 连接表后会检查行数、未匹配记录和键关系。
- [ ] 优先使用向量化操作，不随意逐行循环。
- [ ] 能完成 CSV/Excel 的读取和导出。

不会的细节可以继续补进 `零碎知识补充.ipynb`；完整的 Pandas 数据分析方法保留在本专题中。

## 官方资料

- Pandas 用户指南：https://pandas.pydata.org/docs/user_guide/
- 10 minutes to pandas：https://pandas.pydata.org/docs/user_guide/10min.html
- API Reference：https://pandas.pydata.org/docs/reference/index.html

遇到方法参数不确定时，优先查看与你本机版本一致的官方文档。